# 122 · OpenVLA on LIBERO with five seeds and Wilson intervals (Colab GPU, ~1.5 h on a T4 in 4-bit)

## 1. GPU and disk

In [ ]:
!nvidia-smi
!df -h /content | tail -1

## 2. Python 3.10 environment

In [ ]:
import os
!pip install -q uv
!uv venv /content/venv --python 3.10
!apt-get install -y -qq libgl1 libegl1 libosmesa6 patchelf

os.environ["UV_PYTHON"] = "/content/venv/bin/python"   # every `!uv pip install` targets the venv
PY = "/content/venv/bin/python"
!{PY} --version

## 3. Install packages into the venv

In [ ]:
!uv pip install torch torchvision

In [ ]:
# LIBERO requirements, minus lines that clash with the model stack (torch*, transformers, einops).
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO /content/LIBERO
!grep -viE "^(torch|torchvision|torchaudio|transformers|einops)" /content/LIBERO/requirements.txt > /content/libero_reqs.txt
!uv pip install -r /content/libero_reqs.txt

In [ ]:
# OpenVLA stack + MuJoCo pin (robosuite 1.4.0 breaks on MuJoCo 3.x) + 4-bit loading + video writing.
!uv pip install "transformers==4.40.1" "tokenizers==0.19.1" "timm==0.9.10" accelerate sentencepiece einops huggingface_hub
!uv pip install "mujoco==2.3.7" bitsandbytes imageio imageio-ffmpeg
!uv pip install --no-deps -e /content/LIBERO

## 4. Verify the environment

In [ ]:
%%writefile /content/check_env.py
import torch, transformers, tokenizers, timm, numpy, mujoco, bitsandbytes, imageio
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| tokenizers", tokenizers.__version__, "| timm", timm.__version__,
      "| numpy", numpy.__version__, "| mujoco", mujoco.__version__,
      "| bitsandbytes", bitsandbytes.__version__)
print("cuda available:", torch.cuda.is_available(), "| torch cuda:", torch.version.cuda)
from transformers import AutoProcessor, AutoModelForVision2Seq  # must import cleanly
x = torch.randn(1024, 1024, device="cuda")
print("matmul ok:", (x @ x).sum().item())

In [ ]:
!{PY} /content/check_env.py

## 5. Download the fine-tuned OpenVLA checkpoint (~14 GB)

In [ ]:
%%writefile /content/download_weights.py
from huggingface_hub import snapshot_download
d = snapshot_download(repo_id="openvla/openvla-7b-finetuned-libero-object",
                      local_dir="/content/weights/openvla-7b-finetuned-libero-object")
print("Weights at:", d)

In [ ]:
!{PY} /content/download_weights.py
!ls -la /content/weights/openvla-7b-finetuned-libero-object

## 6. LIBERO config

In [ ]:
import os
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
root = "/content/LIBERO/libero/libero"
cfg = f"""benchmark_root: {root}
bddl_files: {root}/bddl_files
init_states: {root}/init_files
datasets: /content/LIBERO/libero/datasets
assets: {root}/assets
"""
open(os.path.expanduser("~/.libero/config.yaml"), "w").write(cfg)
print(cfg)

## 7. Run OpenVLA in the simulator and record a video

Records the agent camera and the wrist camera side by side to `/content/rollout.mp4`, even if the run crashes partway.
Flags to flip if behaviour looks wrong: `INVERT_GRIPPER`, `ROTATE_180`. Set `TASK_ID` (0-9) to try other `libero_object` tasks.

In [ ]:
%%writefile /content/seed_eval.py
import os, sys, json
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, "/content/LIBERO")
import functools
import numpy as np, torch, imageio
from PIL import Image
import transformers.modeling_utils as _mu
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

MODEL_DIR = "/content/weights/openvla-7b-finetuned-libero-object"
QUANT = "4bit"           # "4bit" (~5 GB), "8bit" (~9 GB), or "bf16" (needs a 24 GB GPU)
TASK_ID = 2              # which libero_object task (0-9)
INIT_STATE_ID = 0        # which fixed starting state for that task
MAX_STEPS = 300
INVERT_GRIPPER = True    # flip to False if the gripper behaves backwards
ROTATE_180 = True        # flip to False to only flip vertically
VIDEO_PATH = "/content/rollout.mp4"

stats = json.load(open(f"{MODEL_DIR}/config.json")).get("norm_stats", {})
UNNORM_KEY = next((k for k in stats if k.startswith("libero_object")), None)
assert UNNORM_KEY, f"no libero_object key; found {list(stats.keys())}"
print("using unnorm_key:", UNNORM_KEY, "| quantization:", QUANT)

# Workaround: new accelerate calls model.to(device) after loading, which old transformers (4.40.1)
# forbids for bitsandbytes models. With a single GPU the weights are already placed, so skip it.
_orig_dispatch = _mu.dispatch_model
def _dispatch_skip_single_device(model, **kw):
    dm = kw.get("device_map")
    if isinstance(dm, dict) and len(set(dm.values())) == 1:
        return model
    return _orig_dispatch(model, **kw)
_mu.dispatch_model = _dispatch_skip_single_device

processor = AutoProcessor.from_pretrained(MODEL_DIR, trust_remote_code=True)
kwargs = dict(torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, trust_remote_code=True)
if QUANT == "4bit":
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16)
    kwargs["device_map"] = {"": 0}
elif QUANT == "8bit":
    kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    kwargs["device_map"] = {"": 0}
model = AutoModelForVision2Seq.from_pretrained(MODEL_DIR, **kwargs)
if QUANT == "bf16":
    model = model.to("cuda")
model.eval()

# The skipped .to() also moved small non-quantized tensors (e.g. rotary inv_freq) to the GPU.
moved_b = moved_p = 0
for mod in model.modules():
    for k, b in list(mod._buffers.items()):
        if b is not None and b.device.type != "cuda":
            mod._buffers[k] = b.to("cuda"); moved_b += 1
    for k, p in list(mod._parameters.items()):
        if p is not None and p.device.type != "cuda":
            p.data = p.data.to("cuda"); moved_p += 1
print(f"moved {moved_b} buffers and {moved_p} params from cpu to cuda", flush=True)
print(f"GPU memory after load: {torch.cuda.memory_allocated()/1e9:.1f} GB", flush=True)


import json as _json
suite = benchmark.get_benchmark_dict()["libero_object"]()
_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)
results = {}
for task_id in range(2):
    task = suite.get_task(task_id)
    bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=256, camera_widths=256)
    init_states = suite.get_task_init_states(task_id)
    instruction = task.language.lower()
    for seed in range(5):
        wins = 0
        for ep in range(10):
            env.seed(seed * 100 + ep); env.reset(); obs = env.set_init_state(init_states[(seed * 10 + ep) % len(init_states)])
            for _ in range(10):
                obs, _, _, _ = env.step([0, 0, 0, 0, 0, 0, -1])
            for step in range(MAX_STEPS):
                image = Image.fromarray(obs["agentview_image"][::-1, ::-1].copy())
                inputs = processor(f"In: What action should the robot take to {instruction}?\nOut:", image).to("cuda", dtype=torch.bfloat16)
                a = np.array(model.predict_action(**inputs, unnorm_key=UNNORM_KEY, do_sample=False), dtype=np.float64)
                a[-1] = -(1.0 if a[-1] > 0.5 else -1.0)
                obs, r, done, info = env.step(a.tolist())
                if done:
                    wins += int(r > 0); break
        results[f"task{task_id}_seed{seed}"] = [wins, 10]
        print(task_id, seed, f"{wins}/10", flush=True)
    env.close()
torch.load = _orig_load
_json.dump(results, open("/content/seed_results.json", "w"))


In [ ]:
!{PY} /content/seed_eval.py 2>&1 | grep -vE "Warning|warn|WARNING|Gym|gymnasium"

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from scipy.stats import norm
def wilson(k, n, conf=0.95):
    z = norm.ppf(1 - (1 - conf) / 2); p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z / d * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return c - h, c + h
res = json.load(open("/content/seed_results.json"))
for task in sorted({k.split("_")[0] for k in res}):
    ks = [res[k][0] for k in res if k.startswith(task)]
    for i, k in enumerate(ks):
        lo, hi = wilson(k, 10); print(f"{task} seed {i}: {k}/10  CI [{lo:.0%}, {hi:.0%}]")
    K, N = sum(ks), 10 * len(ks); lo, hi = wilson(K, N); print(f"{task} pooled: {K}/{N} = {K / N:.0%}  CI [{lo:.0%}, {hi:.0%}]\n")
